In [10]:
# Librerias
import os
import json
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient, DocumentIntelligenceAdministrationClient
from azure.core.exceptions import HttpResponseError
import requests
from azure.identity import DefaultAzureCredential


# Recargar variables del archivo .env
load_dotenv(override=True)


True

In [11]:
# Document Intelligence - Base Models
endpoint = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT")

credential = DefaultAzureCredential()
client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=credential,
)



In [12]:

# Resultado como markdown
def resultado_a_markdown(resultado, titulo="Documento procesado"):
    """
    Convierte el resultado de Azure AI Document Intelligence
    (por ejemplo, prebuilt-layout) a Markdown, incluyendo tablas.

    Args:
        resultado: Resultado de poller.result().
        titulo: Título del documento.

    Returns:
        str: Documento Markdown.
    """
    lineas_md = [
        f"# {titulo}",
        "",
        f"**Modelo utilizado:** `{resultado.model_id}`",
        f"**Número de páginas:** {len(resultado.pages or [])}",
        f"**Tablas detectadas:** {len(resultado.tables or [])}",
        "",
        "---",
        "",
    ]

    def limpiar(texto):
        """Evita que caracteres del contenido rompan la tabla Markdown."""
        if not texto:
            return ""
        return (
            str(texto)
            .replace("|", r"\|")
            .replace("\r", " ")
            .replace("\n", " ")
            .strip()
        )

    def tabla_a_markdown(tabla):
        """Convierte una tabla de Document Intelligence a Markdown."""
        filas = [
            [""] * tabla.column_count
            for _ in range(tabla.row_count)
        ]

        for celda in tabla.cells:
            filas[celda.row_index][celda.column_index] = limpiar(
                celda.content
            )

        # Elimina filas completamente vacías al principio y al final.
        while filas and not any(filas[0]):
            filas.pop(0)

        while filas and not any(filas[-1]):
            filas.pop()

        if not filas:
            return ""

        # Completa las filas para asegurar el mismo número de columnas.
        ancho = max(len(fila) for fila in filas)
        filas = [fila + [""] * (ancho - len(fila)) for fila in filas]

        salida = []
        salida.append("| " + " | ".join(filas[0]) + " |")
        salida.append("| " + " | ".join(["---"] * ancho) + " |")

        for fila in filas[1:]:
            salida.append("| " + " | ".join(fila) + " |")

        return "\n".join(salida)

    # Organiza las tablas por página de aparición, usando bounding_regions.
    tablas_por_pagina = {}

    for indice, tabla in enumerate(resultado.tables or [], start=1):
        regiones = getattr(tabla, "bounding_regions", None) or []

        paginas_tabla = {
            region.page_number for region in regiones
        } or {None}

        for numero_pagina in paginas_tabla:
            tablas_por_pagina.setdefault(numero_pagina, []).append(
                (indice, tabla)
            )

    for pagina in resultado.pages or []:
        numero_pagina = pagina.page_number

        lineas_md.extend([
            f"## Página {numero_pagina}",
            "",
        ])

        # Obtiene las tablas asociadas a esta página.
        tablas_pagina = tablas_por_pagina.get(numero_pagina, [])
        tablas_pagina_ids = {indice for indice, _ in tablas_pagina}

        # Texto detectado en la página.
        if pagina.lines:
            for linea in pagina.lines:
                texto = limpiar(linea.content)
                if texto:
                    lineas_md.extend([texto, ""])

        else:
            lineas_md.extend([
                "*No se detectó texto en esta página.*",
                "",
            ])

        # Muestra las tablas estructuradas detectadas en la página.
        if tablas_pagina:
            lineas_md.extend([
                f"### Tablas de la página {numero_pagina}",
                "",
            ])

            for indice, tabla in tablas_pagina:
                # Evita duplicar una tabla que aparece en varias páginas.
                if indice in getattr(
                    resultado_a_markdown, "_tablas_mostradas", set()
                ):
                    continue

                lineas_md.extend([
                    f"**Tabla {indice}**",
                    "",
                    tabla_a_markdown(tabla),
                    "",
                ])

                # El seguimiento real se gestiona fuera de la función.
                # Se actualiza el conjunto local al final de cada tabla.

    return "\n".join(lineas_md)



# Modelos disponibles

In [13]:
admin_client = DocumentIntelligenceAdministrationClient(
    endpoint=os.environ["DOCUMENT_INTELLIGENCE_ENDPOINT"],
    credential=credential
)

for modelo in admin_client.list_models():
    print("ID:", modelo.model_id,"| Descripción:", modelo.description)

ID: prebuilt-bankStatement.us | Descripción: Extract key information from US bank statement.
ID: prebuilt-check.us | Descripción: Extract key information from United States check.
ID: prebuilt-contract | Descripción: Extract key information from legal contracts.
ID: prebuilt-creditCard | Descripción: Extract key information from credit card.
ID: prebuilt-healthInsuranceCard.us | Descripción: Extract key information from US health insurance cards.
ID: prebuilt-idDocument | Descripción: Extract key information from passports and ID cards.
ID: prebuilt-invoice | Descripción: Extract key information from invoices.
ID: prebuilt-layout | Descripción: Extract text and layout information from documents.
ID: prebuilt-marriageCertificate.us | Descripción: Extract key information from US marriage certificate.
ID: prebuilt-mortgage.us.1003 | Descripción: Extract key information from US Uniform Residential Loan Application (Form 1003).
ID: prebuilt-mortgage.us.1004 | Descripción: Extract key inform

# Modelo: prebuilt-layout

In [14]:
# Ejemplo con documento doc_ejemplo1.pdf
ruta_documento = "ocr_examples/guia_tarjetas_credito_banco_andino_horizonte.pdf"

# Ejemplo usando prebuilt-read
with open(ruta_documento, "rb") as archivo:
    poller = client.begin_analyze_document(
        model_id="prebuilt-layout",
        body=archivo,
        content_type="application/octet-stream",
    )

resultado = poller.result()

print("Análisis completado")
print("Modelo utilizado:", resultado.model_id)
print("Número de páginas:", len(resultado.pages))

# Se muestra el resultado
markdown = resultado_a_markdown(resultado,titulo="Documento 1")
print(markdown)

Análisis completado
Modelo utilizado: prebuilt-layout
Número de páginas: 3
# Documento 1

**Modelo utilizado:** `prebuilt-layout`
**Número de páginas:** 3
**Tablas detectadas:** 4

---

## Página 1

Guía de Tarjetas de Crédito \| Documento ficticio

BANCO ANDINO HORIZONTE

Guía de Tarjetas de Crédito

Productos, beneficios y condiciones para clientes actuales y nuevos solicitantes

Banco Andino Horizonte ofrece cinco tarjetas de crédito diseñadas para distintos perfiles: desde quienes inician su historial

crediticio hasta clientes con alto nivel de consumo y viajes frecuentes. Todas las tarjetas incluyen la billetera digital,

compras por internet, pagos sin contacto y alertas por consumo en tiempo real. Los montos están expresados en soles

(S/) salvo indicación contraria.

1. Tipos de tarjetas disponibles

Tarjeta

Perfil ideal

Línea de crédito

Marca

Programa de recompensas

Andino Clásica

Primera tarjeta o historial

crediticio en construcción

S/ 1,000 - S/ 8,000

Visa

1 punt

# Modelo: prebuilt-invoice

In [15]:
# Ejemplo con documento doc_ejemplo2.pdf, ejemplo de formato pre construido para facturas por microsoft
ruta_documento = "ocr_examples/factura_ficticia.pdf"

# Ejemplo usando prebuilt-invoice
with open(ruta_documento, "rb") as archivo:
    poller = client.begin_analyze_document(
        model_id="prebuilt-invoice",
        body=archivo,
        content_type="application/octet-stream",
    )

invoices = poller.result()

json_pretty = json.dumps(invoices.as_dict(), indent=6)
print(json_pretty)

{
      "apiVersion": "2024-11-30",
      "modelId": "prebuilt-invoice",
      "stringIndexType": "textElements",
      "content": "COMERCIAL QUILLCA S.A.C.\nFACTURA ELECTR\u00d3NICA\nVenta de tecnolog\u00eda y accesorios para el hogar\nAv. Los Alamos 1250, Of. 304, Surquillo, Lima\nR.U.C. 20999999991\nTel: (01) 555-0142 | ventas@quillca.example\nF001 - 00004821\nFecha de emisi\u00f3n:\n05/11/2026\nMoneda:\nSoles (PEN)\nCliente:\nMar\u00eda Fernanda Rojas Quispe\nCondici\u00f3n de pago:\nContado\nDNI / RUC:\nDNI 45678912\nForma de pago:\nTarjeta de cr\u00e9dito \u00b7\u00b7\u00b7\u00b7 4821\nDirecci\u00f3n:\nCalle Las Orqu\u00eddeas 340,\nOrden de compra:\nOC-2026-0873\nMiraflores, Lima\nC\u00f3d.\nDescripci\u00f3n\nUnd.\nCant.\nV. Unitario\nImporte\nTEC-0012\nLaptop UltraLite 14\" 16GB / 512GB SSD\nUND\n1\n2,542.37\n2,542.37\nACC-0207\nMouse inal\u00e1mbrico ergon\u00f3mico\nUND\n2\n59.32\n118.64\nACC-0310\nMochila antirrobo para laptop 15.6\"\nUND\n1\n118.64\n118.64\nACC-0455\nCable 

In [16]:

if invoices.documents:
    for idx, invoice in enumerate(invoices.documents):
        print(f"--------Analyzing invoice #{idx + 1}--------")
        vendor_name = invoice.fields.get("VendorName")
        if vendor_name:
            print(
                f"Vendor Name: {vendor_name.get('content')} has confidence: {vendor_name.get('confidence')}"
            )
        vendor_address = invoice.fields.get("VendorAddress")
        if vendor_address:
            print(
                f"Vendor Address: {vendor_address.get('content')} has confidence: {vendor_address.get('confidence')}"
            )
        vendor_address_recipient = invoice.fields.get("VendorAddressRecipient")
        if vendor_address_recipient:
            print(
                f"Vendor Address Recipient: {vendor_address_recipient.get('content')} has confidence: {vendor_address_recipient.get('confidence')}"
            )
        customer_name = invoice.fields.get("CustomerName")
        if customer_name:
            print(
                f"Customer Name: {customer_name.get('content')} has confidence: {customer_name.get('confidence')}"
            )
        customer_id = invoice.fields.get("CustomerId")
        if customer_id:
            print(
                f"Customer Id: {customer_id.get('content')} has confidence: {customer_id.get('confidence')}"
            )
        customer_address = invoice.fields.get("CustomerAddress")
        if customer_address:
            print(
                f"Customer Address: {customer_address.get('content')} has confidence: {customer_address.get('confidence')}"
            )
        customer_address_recipient = invoice.fields.get("CustomerAddressRecipient")
        if customer_address_recipient:
            print(
                f"Customer Address Recipient: {customer_address_recipient.get('content')} has confidence: {customer_address_recipient.get('confidence')}"
            )
        invoice_id = invoice.fields.get("InvoiceId")
        if invoice_id:
            print(
                f"Invoice Id: {invoice_id.get('content')} has confidence: {invoice_id.get('confidence')}"
            )
        invoice_date = invoice.fields.get("InvoiceDate")
        if invoice_date:
            print(
                f"Invoice Date: {invoice_date.get('content')} has confidence: {invoice_date.get('confidence')}"
            )
        invoice_total = invoice.fields.get("InvoiceTotal")
        if invoice_total:
            print(
                f"Invoice Total: {invoice_total.get('content')} has confidence: {invoice_total.get('confidence')}"
            )
        due_date = invoice.fields.get("DueDate")
        if due_date:
            print(
                f"Due Date: {due_date.get('content')} has confidence: {due_date.get('confidence')}"
            )
        purchase_order = invoice.fields.get("PurchaseOrder")
        if purchase_order:
            print(
                f"Purchase Order: {purchase_order.get('content')} has confidence: {purchase_order.get('confidence')}"
            )
        billing_address = invoice.fields.get("BillingAddress")
        if billing_address:
            print(
                f"Billing Address: {billing_address.get('content')} has confidence: {billing_address.get('confidence')}"
            )
        billing_address_recipient = invoice.fields.get("BillingAddressRecipient")
        if billing_address_recipient:
            print(
                f"Billing Address Recipient: {billing_address_recipient.get('content')} has confidence: {billing_address_recipient.get('confidence')}"
            )
        shipping_address = invoice.fields.get("ShippingAddress")
        if shipping_address:
            print(
                f"Shipping Address: {shipping_address.get('content')} has confidence: {shipping_address.get('confidence')}"
            )
        shipping_address_recipient = invoice.fields.get("ShippingAddressRecipient")
        if shipping_address_recipient:
            print(
                f"Shipping Address Recipient: {shipping_address_recipient.get('content')} has confidence: {shipping_address_recipient.get('confidence')}"
            )
        print("Invoice items:")
        for idx, item in enumerate(invoice.fields.get("Items").get("valueArray")):
            print(f"...Item #{idx + 1}")
            item_description = item.get("valueObject").get("Description")
            if item_description:
                print(
                    f"......Description: {item_description.get('content')} has confidence: {item_description.get('confidence')}"
                )
            item_quantity = item.get("valueObject").get("Quantity")
            if item_quantity:
                print(
                    f"......Quantity: {item_quantity.get('content')} has confidence: {item_quantity.get('confidence')}"
                )
            unit = item.get("valueObject").get("Unit")
            if unit:
                print(
                    f"......Unit: {unit.get('content')} has confidence: {unit.get('confidence')}"
                )
            unit_price = item.get("valueObject").get("UnitPrice")
            if unit_price:
                unit_price_code = (
                    unit_price.get("valueCurrency").get("currencyCode")
                    if unit_price.get("valueCurrency").get("currencyCode")
                    else ""
                )
                print(
                    f"......Unit Price: {unit_price.get('content')}{unit_price_code} has confidence: {unit_price.get('confidence')}"
                )
            product_code = item.get("valueObject").get("ProductCode")
            if product_code:
                print(
                    f"......Product Code: {product_code.get('content')} has confidence: {product_code.get('confidence')}"
                )
            item_date = item.get("valueObject").get("Date")
            if item_date:
                print(
                    f"......Date: {item_date.get('content')} has confidence: {item_date.get('confidence')}"
                )
            tax = item.get("valueObject").get("Tax")
            if tax:
                print(
                    f"......Tax: {tax.get('content')} has confidence: {tax.get('confidence')}"
                )
            amount = item.get("valueObject").get("Amount")
            if amount:
                print(
                    f"......Amount: {amount.get('content')} has confidence: {amount.get('confidence')}"
                )
        subtotal = invoice.fields.get("SubTotal")
        if subtotal:
            print(
                f"Subtotal: {subtotal.get('content')} has confidence: {subtotal.get('confidence')}"
            )
        total_tax = invoice.fields.get("TotalTax")
        if total_tax:
            print(
                f"Total Tax: {total_tax.get('content')} has confidence: {total_tax.get('confidence')}"
            )
        previous_unpaid_balance = invoice.fields.get("PreviousUnpaidBalance")
        if previous_unpaid_balance:
            print(
                f"Previous Unpaid Balance: {previous_unpaid_balance.get('content')} has confidence: {previous_unpaid_balance.get('confidence')}"
            )
        amount_due = invoice.fields.get("AmountDue")
        if amount_due:
            print(
                f"Amount Due: {amount_due.get('content')} has confidence: {amount_due.get('confidence')}"
            )
        service_start_date = invoice.fields.get("ServiceStartDate")
        if service_start_date:
            print(
                f"Service Start Date: {service_start_date.get('content')} has confidence: {service_start_date.get('confidence')}"
            )
        service_end_date = invoice.fields.get("ServiceEndDate")
        if service_end_date:
            print(
                f"Service End Date: {service_end_date.get('content')} has confidence: {service_end_date.get('confidence')}"
            )
        service_address = invoice.fields.get("ServiceAddress")
        if service_address:
            print(
                f"Service Address: {service_address.get('content')} has confidence: {service_address.get('confidence')}"
            )
        service_address_recipient = invoice.fields.get("ServiceAddressRecipient")
        if service_address_recipient:
            print(
                f"Service Address Recipient: {service_address_recipient.get('content')} has confidence: {service_address_recipient.get('confidence')}"
            )
        remittance_address = invoice.fields.get("RemittanceAddress")
        if remittance_address:
            print(
                f"Remittance Address: {remittance_address.get('content')} has confidence: {remittance_address.get('confidence')}"
            )
        remittance_address_recipient = invoice.fields.get(
            "RemittanceAddressRecipient"
        )
        if remittance_address_recipient:
            print(
                f"Remittance Address Recipient: {remittance_address_recipient.get('content')} has confidence: {remittance_address_recipient.get('confidence')}"
            )

        print("----------------------------------------")

--------Analyzing invoice #1--------
Vendor Name: COMERCIAL QUILLCA S.A.C. has confidence: 0.89
Vendor Address: Av. Los Alamos 1250, Of. 304, Surquillo, Lima has confidence: 0.872
Vendor Address Recipient: COMERCIAL QUILLCA S.A.C. has confidence: 0.89
Customer Name: María Fernanda Rojas Quispe has confidence: 0.908
Customer Address: Calle Las Orquídeas 340,
Miraflores, Lima has confidence: 0.89
Customer Address Recipient: María Fernanda Rojas Quispe has confidence: 0.908
Invoice Id: F001 - 00004821 has confidence: 0.736
Invoice Date: 05/11/2026 has confidence: 0.942
Invoice Total: S/ 3,500.00 has confidence: 0.915
Purchase Order: OC-2026-0873 has confidence: 0.927
Invoice items:
...Item #1
......Description: Laptop UltraLite 14" 16GB / 512GB SSD has confidence: 0.943
......Quantity: 1 has confidence: 0.949
......Unit: UND has confidence: 0.899
......Unit Price: 2,542.37PEN has confidence: 0.946
......Product Code: TEC-0012 has confidence: 0.949
......Amount: 2,542.37 has confidence: 0.

# Guardar en formato JSON

In [17]:
import json
import re
from collections import Counter
from datetime import datetime, timezone
from html.parser import HTMLParser
from pathlib import Path

from azure.ai.documentintelligence.models import DocumentContentFormat

# ----------------------------------------------------------------------
# 1. CONFIGURACIÓN
# ----------------------------------------------------------------------
RUTA_PDF = "ocr_examples/guia_tarjetas_credito_banco_andino_horizonte.pdf"
RUTA_JSON = "salida/guia_tarjetas_credito.json"
URL_DOCUMENTO = "https://empresa.example/manuales/guia_tarjetas_credito.pdf"
AREA = "Banca de Personas"
VIGENTE = True

# Categoría por número de sección (la sección 0 es la introducción)
CATEGORIAS = {
    0: "General",
    1: "Productos",
    2: "Beneficios",
    3: "Requisitos",
    4: "Costos y tarifas",
    5: "Operaciones",
    6: "Procesos",
    7: "Términos y condiciones",
    8: "Restricciones",
    9: "Seguridad",
    10: "Preguntas frecuentes",
}


# ----------------------------------------------------------------------
# 2. UTILIDADES
# ----------------------------------------------------------------------
class _TablaHTML(HTMLParser):
    """Convierte <table> de Document Intelligence a tabla Markdown."""

    def __init__(self):
        super().__init__()
        self.filas, self.fila, self.celda = [], None, None

    def handle_starttag(self, tag, attrs):
        if tag == "tr":
            self.fila = []
        elif tag in ("td", "th"):
            self.celda = []
        elif tag == "br" and self.celda is not None:
            self.celda.append(" ")

    def handle_endtag(self, tag):
        if tag in ("td", "th") and self.celda is not None:
            texto = " ".join("".join(self.celda).split()).replace("|", r"\|")
            self.fila.append(texto)
            self.celda = None
        elif tag == "tr" and self.fila is not None:
            self.filas.append(self.fila)
            self.fila = None

    def handle_data(self, data):
        if self.celda is not None:
            self.celda.append(data)


def tabla_html_a_markdown(html):
    p = _TablaHTML()
    p.feed(html)
    filas = [f for f in p.filas if any(f)]
    if not filas:
        return ""
    ancho = max(len(f) for f in filas)
    filas = [f + [""] * (ancho - len(f)) for f in filas]
    out = ["| " + " | ".join(filas[0]) + " |", "| " + " | ".join(["---"] * ancho) + " |"]
    out += ["| " + " | ".join(f) + " |" for f in filas[1:]]
    return "\n".join(out)


def limpiar_markdown(texto, lineas_repetidas):
    """Quita comentarios de DI, encabezados/pies repetidos y convierte tablas."""
    texto = re.sub(r"<!--.*?-->", "", texto, flags=re.S)
    texto = re.sub(
        r"<table>.*?</table>",
        lambda m: "\n" + tabla_html_a_markdown(m.group(0)) + "\n",
        texto,
        flags=re.S,
    )
    lineas = [l for l in texto.splitlines() if l.strip() not in lineas_repetidas]
    texto = "\n".join(lineas)
    return re.sub(r"\n{3,}", "\n\n", texto).strip()


def detectar_lineas_repetidas(resultado):
    """Líneas que aparecen en todas las páginas (encabezados y pies de página)."""
    n = len(resultado.pages or [])
    if n < 2:
        return set()
    cont = Counter()
    for pagina in resultado.pages:
        for l in {l.content.strip() for l in (pagina.lines or [])}:
            cont[l] += 1
    return {l for l, c in cont.items() if c >= n}


def pagina_de_offset(resultado, offset):
    for pagina in resultado.pages or []:
        for span in pagina.spans or []:
            if span.offset <= offset < span.offset + span.length:
                return pagina.page_number
    return None


def dividir_en_secciones(contenido):
    """Divide por encabezados '1. Título', '2. Título'... (numeración consecutiva)."""
    patron = re.compile(r"^[ \t]*(?:#{1,6}[ \t]*)?(\d{1,2})\.[ \t]+(\S.*?)[ \t]*$", re.M)
    hitos, esperado = [], 1
    for m in patron.finditer(contenido):
        if int(m.group(1)) == esperado:
            hitos.append((esperado, m.group(2).strip(), m.start(), m.end()))
            esperado += 1

    secciones = []
    inicio_intro = hitos[0][2] if hitos else len(contenido)
    secciones.append((0, "Introducción", 0, contenido[:inicio_intro]))
    for i, (num, titulo, ini, fin_titulo) in enumerate(hitos):
        fin = hitos[i + 1][2] if i + 1 < len(hitos) else len(contenido)
        secciones.append((num, titulo, ini, contenido[fin_titulo:fin]))
    return secciones


# ----------------------------------------------------------------------
# 3. EXTRACCIÓN CON DOCUMENT INTELLIGENCE
# ----------------------------------------------------------------------
with open(RUTA_PDF, "rb") as f:
    poller = client.begin_analyze_document(
        model_id="prebuilt-layout",
        body=f,
        content_type="application/octet-stream",
        output_content_format=DocumentContentFormat.MARKDOWN,
    )
resultado = poller.result()
contenido = resultado.content
print("Páginas:", len(resultado.pages), "| Tablas:", len(resultado.tables or []))

# ----------------------------------------------------------------------
# 4. METADATA GENERAL DEL DOCUMENTO
# ----------------------------------------------------------------------
m_version = re.search(r"Versi[oó]n\s+(\d{4})\.(\d{2})", contenido)
version = f"{m_version.group(1)}.{m_version.group(2)}" if m_version else None
fecha_pub = (
    f"{m_version.group(1)}-{m_version.group(2)}-01T00:00:00Z"
    if m_version
    else datetime.now(timezone.utc).strftime("%Y-%m-%dT00:00:00Z")
)

m_tel = re.search(r"Central de atenci[oó]n:\s*(\(?\d+\)?\s*[\d-]+)", contenido)
m_mail = re.search(r"[\w.+-]+@[\w-]+\.[\w.]+", contenido)
m_wsp = re.search(r"WhatsApp[^:]*:\s*(\+?\d[\d ]+\d)", contenido)

confianzas = [
    w.confidence
    for p in resultado.pages
    for w in (p.words or [])
    if w.confidence is not None
]

meta_doc = {
    "documento_id": "guia-tarjetas-credito-banco-andino-horizonte",
    "nombre_archivo": Path(RUTA_PDF).name,
    "entidad": "Banco Andino Horizonte",
    "tipo_documento": "Guía de producto",
    "version": version,
    "idioma": "es",
    "moneda_referencia": "PEN",
    "total_paginas": len(resultado.pages),
    "total_tablas": len(resultado.tables or []),
    "confianza_promedio_ocr": round(sum(confianzas) / len(confianzas), 4) if confianzas else None,
    "modelo_extraccion": resultado.model_id,
    "api_version": resultado.api_version,
    "contacto": {
        "central_atencion": m_tel.group(1).strip() if m_tel else None,
        "whatsapp": m_wsp.group(1).strip() if m_wsp else None,
        "correo": m_mail.group(0) if m_mail else None,
    },
    "es_ficticio": "documento ficticio" in contenido.lower(),
    "fecha_extraccion": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}

# ----------------------------------------------------------------------
# 5. ARMAR REGISTROS
# ----------------------------------------------------------------------
repetidas = detectar_lineas_repetidas(resultado)
registros = []

for num, titulo, offset, texto in dividir_en_secciones(contenido):
    texto_limpio = limpiar_markdown(texto, repetidas)
    if not texto_limpio:
        continue
    pag_ini = pagina_de_offset(resultado, offset)
    pag_fin = pagina_de_offset(resultado, max(offset + len(texto) - 1, offset))
    registros.append(
        {
            "id": str(num + 1),
            "titulo": titulo if num else "Guía de Tarjetas de Crédito - Introducción",
            "contenido": texto_limpio,
            "categoria": CATEGORIAS.get(num, "General"),
            "area": AREA,
            "fecha_publicacion": fecha_pub,
            "vigente": VIGENTE,
            "url_documento": URL_DOCUMENTO,
            "metadata": {
                **meta_doc,
                "seccion_numero": num,
                "pagina_inicio": pag_ini,
                "pagina_fin": pag_fin,
                "num_caracteres": len(texto_limpio),
                "contiene_tablas": "| --- |" in texto_limpio,
            },
        }
    )

# ----------------------------------------------------------------------
# 6. GUARDAR JSON
# ----------------------------------------------------------------------
Path(RUTA_JSON).parent.mkdir(parents=True, exist_ok=True)
with open(RUTA_JSON, "w", encoding="utf-8") as f:
    json.dump(registros, f, ensure_ascii=False, indent=2)

print(f"JSON guardado en {RUTA_JSON} con {len(registros)} registros")
for r in registros:
    print(f"  [{r['id']}] {r['titulo']} (págs {r['metadata']['pagina_inicio']}-{r['metadata']['pagina_fin']}, {r['metadata']['num_caracteres']} car.)")

Páginas: 3 | Tablas: 4
JSON guardado en salida/guia_tarjetas_credito.json con 11 registros
  [1] Guía de Tarjetas de Crédito - Introducción (págs 1-1, 510 car.)
  [2] Tipos de tarjetas disponibles (págs 1-1, 750 car.)
  [3] Beneficios principales por producto (págs 1-2, 1343 car.)
  [4] Requisitos para solicitar una tarjeta (págs 2-2, 1107 car.)
  [5] Costos, tasas y comisiones (págs 2-2, 904 car.)
  [6] Fechas de corte y de pago (págs 2-2, 392 car.)
  [7] Proceso de apertura (págs 2-3, 253 car.)
  [8] Términos y condiciones generales (págs 3-3, 1031 car.)
  [9] Restricciones y exclusiones (págs 3-3, 789 car.)
  [10] Seguridad y protección del cliente (págs 3-3, 431 car.)
  [11] Preguntas frecuentes (págs 3-3, 1134 car.)


#### Considero util el uso de Document Inteligence para tratar informacion adicional como documentos, tablas, etc., relacionada al producto tarjetas de credito. Esto nos ayudara a que mi asistente tenga mayor soporte en su respuesta al cliente. Asimismo se debe seguir probando alternativas de como trabajar cualquier tipo de información adicional en diferentes fuentes. 